In [ ]:
from io import BytesIO
import math
from pathlib import Path
from urllib.parse import quote

import ipywidgets as widgets
import pandas

from IPython.display import display
from matplotlib import pyplot

from searchlight.lightcurves import load
from searchlight.models.simulens.v1 import (
    DEFAULT_SPLIT_SIZES,
    _segment_lightcurve,
    _segment_windows_for_split_size,
)
from searchlight.review_exports import (
    DEFAULT_METADATA_CACHE_PATH,
    DEFAULT_RESULT_CACHE_META_PATH,
    DEFAULT_RESULT_CACHE_PATH,
    RESULT_COLUMNS,
    RESULT_DTYPES,
    enrich_results_with_metadata,
    load_result_frames,
)
from searchlight.zooniverse_results import (
    DEFAULT_SUBJECT_METADATA_CACHE_PATH,
    DEFAULT_TASK_KEY,
    add_candidate_ids,
    annotate_most_likely_answers,
    load_zooniverse_results,
    select_zooniverse_answer_results,
)


In [ ]:
RESULTS_PATH = Path('data/results/simulens1.0_tess_spoc')
RESULT_CACHE_PATH = DEFAULT_RESULT_CACHE_PATH
RESULT_CACHE_META_PATH = DEFAULT_RESULT_CACHE_META_PATH
RESULT_METADATA_CACHE_PATH = DEFAULT_METADATA_CACHE_PATH

# CSV exported from Zooniverse/aggregation, containing integer subject_id and data.most_likely.
ZOONIVERSE_RESULTS_CSV = Path('data/zooniverse/check_peaks_results.csv')
SUBJECT_METADATA_CACHE_PATH = DEFAULT_SUBJECT_METADATA_CACHE_PATH

# Set this explicitly if the CSV does not contain a workflow_id column.
WORKFLOW_ID = None
TASK_KEY = DEFAULT_TASK_KEY
MOST_LIKELY_ANSWER = 'Three or more peaks'
TOP_N = 100


In [ ]:
# Zooniverse CSV loading, workflow answer lookup, and subject metadata mapping
# live in searchlight.zooniverse_results so this notebook and the exporter use
# the same implementation.


In [ ]:
def make_review_item(row, lc):
    path = row['path']
    score = row.get('lensing')
    subject_id = row.get('subject_id')
    most_likely_answer = row.get('most_likely_answer')
    num_votes = row.get('data.num_votes')
    agreement = row.get('data.agreement')

    header = widgets.HBox([
        widgets.HTML(f'<strong>{Path(path).name}</strong>'),
        widgets.HTML(f'<code>subject_id={subject_id}</code>'),
        widgets.HTML(f'<code>most_likely={most_likely_answer}</code>'),
    ])

    ticid = lc.meta.get('TICID') or lc.meta.get('TARGETID')
    sector = lc.meta.get('SECTOR')
    ra = lc.meta.get('RA')
    dec = lc.meta.get('DEC')

    lines = [
        f'<div><strong>File:</strong> <code>{path}</code></div>',
        f'<div><strong>Subject ID:</strong> <code>{subject_id}</code></div>',
        f'<div><strong>Most likely answer:</strong> {most_likely_answer}</div>',
        f'<div><strong>Votes:</strong> {num_votes}</div>',
        f'<div><strong>Agreement:</strong> {agreement}</div>',
    ]
    if pandas.notna(score):
        lines.append(f'<div><strong>Simulens score:</strong> {score}</div>')
    if ticid is not None:
        lines.append(f'<div><strong>TIC:</strong> {ticid}</div>')
    if sector is not None:
        lines.append(f'<div><strong>Sector:</strong> {sector}</div>')
    if ra is not None and dec is not None:
        coord = f'{ra} {dec}'
        simbad_url = 'https://simbad.cds.unistra.fr/simbad/sim-coo?Coord=' + quote(coord)
        lines.append(f'<div><strong>SIMBAD:</strong> <a href="{simbad_url}" target="_blank">{simbad_url}</a></div>')
    details = widgets.HTML(''.join(lines))

    split_children = []
    open_index = None

    def split_label(split_size_days):
        if math.isinf(split_size_days):
            return 'all data'
        return f'{split_size_days:g} days'

    for split_size in DEFAULT_SPLIT_SIZES:
        split_size_days = float(split_size)
        windows = _segment_windows_for_split_size(lc, split_size_days)
        segments = [
            _segment_lightcurve(lc, start, end)
            for start, end, _ in windows
        ]
        segments = [seg for seg in segments if len(seg) > 0]

        if not segments:
            split_children.append(widgets.HTML(f'<div>No non-empty segments for <code>{split_label(split_size_days)}</code>.</div>'))
            if math.isinf(split_size_days):
                open_index = len(split_children) - 1
            continue

        fig, axes = pyplot.subplots(
            len(segments),
            1,
            figsize=(12, max(2.5, 2.5 * len(segments))),
            squeeze=False,
        )
        axes = axes.flatten()
        fig.suptitle(f'{Path(path).name} | split_size={split_size_days} days', fontsize=12)

        for i, seg in enumerate(segments):
            ax = axes[i]
            ax.scatter(seg['time'].jd, seg['flux'], s=2, alpha=0.6)
            ax.set_ylabel('flux')
            ax.set_title(f'segment {i + 1} ({len(seg)} points)', fontsize=10)

        axes[-1].set_xlabel('time (JD)')
        pyplot.tight_layout()

        buffer = BytesIO()
        fig.savefig(buffer, format='png', bbox_inches='tight')
        pyplot.close(fig)
        split_children.append(widgets.Image(value=buffer.getvalue(), format='png'))
        if math.isinf(split_size_days):
            open_index = len(split_children) - 1

    plots = widgets.Accordion(children=split_children)
    for index, split_size in enumerate(DEFAULT_SPLIT_SIZES):
        plots.set_title(index, split_label(float(split_size)))
    plots.selected_index = open_index

    card = widgets.VBox([header, details, plots])
    card.layout.border = '1px solid #ddd'
    card.layout.padding = '8px'
    card.layout.margin = '0 0 16px 0'
    return card


In [ ]:
result_paths = sorted(RESULTS_PATH.glob('*_resnet.csv'))
results = load_result_frames(
    result_paths,
    RESULT_COLUMNS,
    RESULT_DTYPES,
    cache_path=RESULT_CACHE_PATH,
    cache_meta_path=RESULT_CACHE_META_PATH,
)
results = results[results['label'] == 'lensing'].copy()
results = add_candidate_ids(results)
results.sort_values('lensing', inplace=True, ascending=False)
print(f'Loaded {len(results)} lensing model rows from {len(result_paths)} result CSVs')


In [ ]:
zooniverse = load_zooniverse_results(ZOONIVERSE_RESULTS_CSV)
zooniverse, workflow_id, answer_labels = annotate_most_likely_answers(
    zooniverse,
    workflow_id=WORKFLOW_ID,
    task_key=TASK_KEY,
)
zooniverse['most_likely_answer'].value_counts(dropna=False)


In [ ]:
selected = select_zooniverse_answer_results(
    results,
    zooniverse,
    MOST_LIKELY_ANSWER,
    subject_metadata_cache_path=SUBJECT_METADATA_CACHE_PATH,
)
review_results = selected.head(TOP_N).copy()
review_results[['subject_id', 'most_likely_answer', 'data.num_votes', 'data.agreement', 'candidate_id', 'path', 'lensing']].head()


In [ ]:
review_export = enrich_results_with_metadata(
    review_results,
    cache_path=RESULT_METADATA_CACHE_PATH,
)
review_export[[
    'subject_id',
    'most_likely_answer',
    'data.num_votes',
    'data.agreement',
    'path',
    'lensing',
    'tic',
    'sector',
    'simbad_url',
]].head()


In [ ]:
for row in review_results.to_dict('records'):
    lc = load(row['path'])
    display(make_review_item(row, lc))
